# 02 - Data Quality Assessment

## Objective

Systematically assess the raw e-commerce dataset to identify missing values, duplicate records, inconsistent categorical
values, invalid numerical values, date inconsistencies, and potential outliers.

This notebook focuses exclusively on identifying and documenting data quality issues. No modifications will be made to the original dataset.

## Dataset

Source: `Dataset/Raw/ecommerce_sales_customer.csv`

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../Dataset/Raw/ecommerce_sales_customer.csv")

In [3]:
print(f"Dataset shape: {df.shape}")
print(f"Total records: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")

Dataset shape: (50000, 18)
Total records: 50,000
Total columns: 18


In [4]:
df.head()

,OrderID,OrderDate,CustomerID,CustomerRegistrationDate,ProductID,ProductName,Category,Quantity,UnitPrice,DiscountPct,GrossSales,DiscountAmount,NetSales,Region,Country,PaymentMethod,ReturnStatus,ReturnDate
0,ORD003508,2025-02-25,CUST07492,2024-07-08,P012,Kitchen Organizer,Home & Kitchen,4,33.21,0.0,132.84,0.00,132.84,Europe,Spain,Gift Card,Not Returned,NaN
1,ORD008536,2025-01-25,CUST06763,2022-08-07,P015,Water Bottle,Sports,3,17.01,20.0,51.03,10.21,40.82,Asia Pacific,India,Debit Card,Not Returned,NaN
2,ORD009501,2025-06-16,CUST07225,2023-06-30,P012,Kitchen Organizer,Home & Kitchen,5,35.11,0.0,175.55,0.00,175.55,Europe,Germany,Digital Wallet,Not Returned,NaN
3,ORD015712,2025-11-12,CUST04817,2024-08-18,P016,Face Moisturizer,Beauty,5,25.10,0.0,125.50,0.00,125.50,Latin America,Chile,Cash on Delivery,Not Returned,NaN
4,ORD003485,2025-02-04,CUST00134,2022-08-01,P003,Smart Watch,Electronics,1,110.82,10.0,110.82,11.08,99.74,Europe,Spain,Credit Card,Not Returned,NaN


## 1. Missing Value Assessment

Identify columns containing missing values and evaluate the extent of missing data in the dataset.

In [5]:
missing_values = df.isnull().sum()

missing_values

OrderID                         0
OrderDate                       0
CustomerID                      0
CustomerRegistrationDate        0
ProductID                       0
ProductName                     0
Category                       90
Quantity                        0
UnitPrice                       0
DiscountPct                     0
GrossSales                      0
DiscountAmount                  0
NetSales                        0
Region                         80
Country                         0
PaymentMethod                  79
ReturnStatus                    0
ReturnDate                  46810
dtype: int64

In [6]:
missing_percentage = (df.isnull().mean() * 100).round(2)

missing_percentage

OrderID                      0.00
OrderDate                    0.00
CustomerID                   0.00
CustomerRegistrationDate     0.00
ProductID                    0.00
ProductName                  0.00
Category                     0.18
Quantity                     0.00
UnitPrice                    0.00
DiscountPct                  0.00
GrossSales                   0.00
DiscountAmount               0.00
NetSales                     0.00
Region                       0.16
Country                      0.00
PaymentMethod                0.16
ReturnStatus                 0.00
ReturnDate                  93.62
dtype: float64

In [7]:
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values(by="Missing Percentage", ascending=False)

missing_summary

,Missing Values,Missing Percentage
ReturnDate,46810,93.62
Category,90,0.18
Region,80,0.16
PaymentMethod,79,0.16


### Initial Observations

- ReturnDate contains 46,810 missing values (93.62%), which may be expected for orders that were not returned.
- Category contains 90 missing values.
- Region contains 80 missing values.
- PaymentMethod contains 79 missing values.
- All remaining columns contain no missing values.

Further investigation is required to determine the appropriate treatment of missing values before data cleaning.

## 2. Duplicate Record Assessment

Identify exact duplicate records and assess their impact on the dataset.

In [8]:
duplicate_count = df.duplicated().sum()

print(f"Total duplicate records: {duplicate_count}")

Total duplicate records: 100


In [9]:
duplicate_records = df[df.duplicated(keep=False)]

duplicate_records.head(10)

,OrderID,OrderDate,CustomerID,CustomerRegistrationDate,ProductID,ProductName,Category,Quantity,UnitPrice,DiscountPct,GrossSales,DiscountAmount,NetSales,Region,Country,PaymentMethod,ReturnStatus,ReturnDate
106,ORD017143,2024-05-23,CUST04992,2024-02-06,P009,Cotton T-Shirt,Fashion,1,21.37,10.0,21.37,2.14,19.23,Latin America,Mexico,PayPal,Not Returned,NaN
124,ORD013655,2024-01-28,CUST06975,2022-09-27,P007,Travel Backpack,Accessories,4,68.53,5.0,274.12,13.71,260.41,Asia Pacific,India,Cash on Delivery,Not Returned,NaN
357,ORD010979,2024-05-01,CUST01548,2023-11-18,P020,Cookbook,Books,2,21.48,25.0,42.96,10.74,32.22,Asia Pacific,India,Debit Card,Not Returned,NaN
397,ORD001064,2025-05-05,CUST04718,2022-09-02,P019,Notebook Set,Books,4,13.64,0.0,54.56,0.00,54.56,North America,United States,Digital Wallet,Not Returned,NaN
403,ORD012302,2025-07-18,CUST05848,2021-10-03,P017,Skincare Bundle,Beauty,1,47.38,5.0,47.38,2.37,45.01,Europe,United Kingdom,Cash on Delivery,Not Returned,NaN
426,ORD002233,2025-10-24,CUST06712,2023-10-14,P015,Water Bottle,Sports,1,17.28,20.0,17.28,3.46,13.82,Latin America,Brazil,Credit Card,Not Returned,NaN
506,ORD006402,2024-05-20,CUST03300,2022-02-19,P012,Kitchen Organizer,Home & Kitchen,2,35.50,0.0,71.00,0.00,71.00,Latin America,Chile,Cash on Delivery,Not Returned,NaN
559,ORD015758,2024-09-25,CUST05668,2022-01-11,P006,Laptop Sleeve,Accessories,2,26.57,5.0,53.14,2.66,50.48,Middle East,Pakistan,Debit Card,Not Returned,NaN
747,ORD014407,2025-10-01,CUST02200,2021-12-19,P009,Cotton T-Shirt,Fashion,5,22.26,10.0,111.30,11.13,100.17,Asia Pacific,Singapore,Debit Card,Not Returned,NaN
1197,ORD016908,2025-01-31,CUST04299,2024-07-01,P005,Phone Case,Accessories,1,13.67,5.0,13.67,0.68,12.99,Europe,France,Bank Transfer,Not Returned,NaN


In [10]:
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

Duplicate percentage: 0.20%


## 3. Categorical Consistency Assessment

Examine categorical columns to identify inconsistent text formatting, including differences in capitalization, leading and trailing whitespace, and unexpected category variations.

In [11]:
categorical_columns = df.select_dtypes(include="object").columns

categorical_columns.tolist()

['OrderID',
 'OrderDate',
 'CustomerID',
 'CustomerRegistrationDate',
 'ProductID',
 'ProductName',
 'Category',
 'Region',
 'Country',
 'PaymentMethod',
 'ReturnStatus',
 'ReturnDate']

In [12]:
for column in categorical_columns:
    print(f"{column}: {df[column].nunique()} unique values")

OrderID: 20745 unique values
OrderDate: 912 unique values
CustomerID: 6985 unique values
CustomerRegistrationDate: 1518 unique values
ProductID: 20 unique values
ProductName: 20 unique values
Category: 28 unique values
Region: 6 unique values
Country: 17 unique values
PaymentMethod: 7 unique values
ReturnStatus: 2 unique values
ReturnDate: 903 unique values


In [13]:
print("Category values:")
print(sorted(df["Category"].dropna().unique()))

Category values:
[' Accessories ', ' Beauty ', ' Books ', ' Electronics ', ' Fashion ', ' Home & Kitchen ', ' Sports ', 'ACCESSORIES', 'Accessories', 'BEAUTY', 'BOOKS', 'Beauty', 'Books', 'ELECTRONICS', 'Electronics', 'FASHION', 'Fashion', 'HOME & KITCHEN', 'Home & Kitchen', 'SPORTS', 'Sports', 'accessories', 'beauty', 'books', 'electronics', 'fashion', 'home & kitchen', 'sports']


In [14]:
print("Payment Method values:")
print(sorted(df["PaymentMethod"].dropna().unique()))

Payment Method values:
['Bank Transfer', 'Cash on Delivery', 'Credit Card', 'Debit Card', 'Digital Wallet', 'Gift Card', 'PayPal']


In [15]:
print("Region values:")
print(sorted(df["Region"].dropna().unique()))

Region values:
['Africa', 'Asia Pacific', 'Europe', 'Latin America', 'Middle East', 'North America']


### Initial Observations

- Category contains 28 unique values, although these represent only 7 logical categories.
- The variations are caused by inconsistent capitalization and leading/trailing whitespace.
- PaymentMethod contains 7 distinct values with no visible formatting inconsistencies.
- Region contains 6 distinct values with no visible formatting inconsistencies.
- Other object-type columns include identifiers and date fields, which are not necessarily categorical variables.

Category standardisation will be considered during the data cleaning stage.

## 4. Numerical Data Validation

Examine numerical columns to identify invalid values, unexpected ranges, and potential inconsistencies in quantities, prices, discounts, and sales calculations.

In [16]:
numerical_columns = df.select_dtypes(include=np.number).columns

numerical_columns.tolist()

['Quantity',
 'UnitPrice',
 'DiscountPct',
 'GrossSales',
 'DiscountAmount',
 'NetSales']

In [17]:
df[numerical_columns].describe()

,Quantity,UnitPrice,DiscountPct,GrossSales,DiscountAmount,NetSales
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,3.003940,37.837105,8.682400,113.842272,9.896129,103.946143
std,1.415544,22.724930,7.601848,92.902485,13.873424,85.574558
min,0.000000,-5.000000,0.000000,-20.000000,-2.250000,-73.320000
25%,2.000000,20.890000,0.000000,49.050000,0.000000,44.400000
50%,3.000000,30.050000,10.000000,85.265000,5.380000,77.620000
75%,4.000000,50.740000,15.000000,148.292500,13.352500,135.622500
max,5.000000,114.990000,125.000000,574.850000,366.600000,574.850000


In [18]:
negative_values = (df[numerical_columns] < 0).sum()

negative_values[negative_values > 0]

UnitPrice         10
GrossSales        10
DiscountAmount     5
NetSales          22
dtype: int64

In [19]:
zero_values = (df[numerical_columns] == 0).sum()

zero_values[zero_values > 0]

Quantity             15
DiscountPct       15038
GrossSales           15
DiscountAmount    15049
NetSales             15
dtype: int64

In [20]:
(df["DiscountPct"] > 100).sum()

np.int64(12)

### Initial Observations

- UnitPrice contains 10 negative values.
- GrossSales contains 10 negative values.
- DiscountAmount contains 5 negative values.
- NetSales contains 22 negative values.
- Quantity contains 15 zero values.
- DiscountPct contains 12 values exceeding 100%.
- Zero DiscountPct values are valid and represent transactions without discounts.
- Negative financial values and zero quantities require further investigation.
- Financial calculation consistency will be assessed separately.

These findings will be investigated further before determining appropriate data-cleaning actions.

## 5. Date Validation

Assess date columns for invalid date values and logical inconsistencies between customer registration, order placement, and return dates.

In [21]:
date_columns = [
    "OrderDate",
    "CustomerRegistrationDate",
    "ReturnDate"
]

date_df = df[date_columns].copy()

for column in date_columns:
    date_df[column] = pd.to_datetime(
        date_df[column],
        errors="coerce"
    )

In [22]:
invalid_dates = date_df.isnull().sum()

invalid_dates

OrderDate                       0
CustomerRegistrationDate        0
ReturnDate                  46810
dtype: int64

In [23]:
(
    date_df["CustomerRegistrationDate"] >
    date_df["OrderDate"]
).sum()

np.int64(0)

In [24]:
(
    date_df["ReturnDate"] <
    date_df["OrderDate"]
).sum()

np.int64(0)

In [25]:
(
    (df["ReturnStatus"] == "Returned") &
    (date_df["ReturnDate"].isna())
).sum()

np.int64(0)

In [26]:
(
    (df["ReturnStatus"] == "Not Returned") &
    (date_df["ReturnDate"].notna())
).sum()

np.int64(0)

### Initial Observations

- OrderDate contains no invalid or missing dates.
- CustomerRegistrationDate contains no invalid or missing dates.
- No customer registration date occurs after its order date.
- No return date occurs before its corresponding order date.
- All returned orders have a recorded ReturnDate.
- Non-returned orders do not have a ReturnDate.
- Missing ReturnDate values are consistent with the ReturnStatus field.

No date-related logical inconsistencies were identified.

## 6. Financial Calculation Consistency

Verify the mathematical relationships between Quantity, UnitPrice, DiscountPct, GrossSales, DiscountAmount, and NetSales to identify inconsistent financial calculations.

In [27]:
expected_gross_sales = df["Quantity"] * df["UnitPrice"]

gross_sales_mismatch = ~np.isclose(
    df["GrossSales"],
    expected_gross_sales,
    atol=0.01
)

gross_sales_mismatch.sum()

np.int64(0)

In [28]:
expected_discount = (
    df["GrossSales"] * df["DiscountPct"] / 100
)

discount_mismatch = ~np.isclose(
    df["DiscountAmount"],
    expected_discount,
    atol=0.01
)

discount_mismatch.sum()

np.int64(0)

In [29]:
expected_net_sales = (
    df["GrossSales"] - df["DiscountAmount"]
)

net_sales_mismatch = ~np.isclose(
    df["NetSales"],
    expected_net_sales,
    atol=0.01
)

net_sales_mismatch.sum()

np.int64(0)

In [30]:
financial_validation = pd.DataFrame({
    "Calculation": [
        "GrossSales",
        "DiscountAmount",
        "NetSales"
    ],
    "Mismatched Records": [
        gross_sales_mismatch.sum(),
        discount_mismatch.sum(),
        net_sales_mismatch.sum()
    ]
})

financial_validation

,Calculation,Mismatched Records
0,GrossSales,0
1,DiscountAmount,0
2,NetSales,0


### Initial Observations

- GrossSales calculations match Quantity multiplied by UnitPrice across all records.
- DiscountAmount calculations match GrossSales multiplied by DiscountPct.
- NetSales calculations match GrossSales minus DiscountAmount.
- No financial calculation mismatches were identified.
- Although calculations are mathematically consistent, negative prices, negative financial values, zero quantities, and discounts exceeding 100% still require business-level investigation.

All three financial calculation checks passed.

## 7. Outlier Assessment

Use the Interquartile Range (IQR) method to identify potential statistical outliers in numerical columns.

In [31]:
outlier_columns = [
    "Quantity",
    "UnitPrice",
    "DiscountPct",
    "GrossSales",
    "DiscountAmount",
    "NetSales"
]

In [32]:
outlier_results = []

for column in outlier_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    outlier_results.append({
        "Column": column,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Outlier Count": outliers.sum(),
        "Outlier Percentage": round(
            outliers.mean() * 100, 2
        )
    })

outlier_summary = pd.DataFrame(outlier_results)

outlier_summary

,Column,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage
0,Quantity,-1.00,7.00,0,0.00
1,UnitPrice,-23.89,95.52,1588,3.18
2,DiscountPct,-22.50,37.50,12,0.02
3,GrossSales,-99.81,297.16,2781,5.56
4,DiscountAmount,-20.03,33.38,3108,6.22
5,NetSales,-92.43,272.46,2803,5.61


In [33]:
Q1 = df["UnitPrice"].quantile(0.25)
Q3 = df["UnitPrice"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

df[df["UnitPrice"] > upper_bound][
    ["ProductName", "UnitPrice", "Quantity", "NetSales"]
].head(10)

,ProductName,UnitPrice,Quantity,NetSales
4,Smart Watch,110.82,1,99.74
25,Smart Watch,111.34,1,100.21
30,Smart Watch,95.64,5,478.20
54,Smart Watch,102.99,4,411.96
61,Smart Watch,99.27,3,282.92
64,Smart Watch,102.53,2,205.06
81,Smart Watch,108.80,3,261.12
144,Smart Watch,111.51,2,223.02
170,Smart Watch,97.71,4,332.21
240,Smart Watch,114.22,4,456.88


In [34]:
lower_bound = Q1 - 1.5 * IQR

unit_price_outliers = df[
    (df["UnitPrice"] < lower_bound) |
    (df["UnitPrice"] > upper_bound)
]

unit_price_outliers[
    ["ProductName", "UnitPrice", "Quantity", "NetSales"]
].head(15)

,ProductName,UnitPrice,Quantity,NetSales
4,Smart Watch,110.82,1,99.74
25,Smart Watch,111.34,1,100.21
30,Smart Watch,95.64,5,478.20
54,Smart Watch,102.99,4,411.96
61,Smart Watch,99.27,3,282.92
64,Smart Watch,102.53,2,205.06
81,Smart Watch,108.80,3,261.12
144,Smart Watch,111.51,2,223.02
170,Smart Watch,97.71,4,332.21
240,Smart Watch,114.22,4,456.88


In [35]:
unit_price_outlier_summary = (
    unit_price_outliers
    .groupby("ProductName")
    .agg(
        OutlierRecords=("UnitPrice", "count"),
        MinimumPrice=("UnitPrice", "min"),
        MaximumPrice=("UnitPrice", "max")
    )
    .sort_values(
        by="OutlierRecords",
        ascending=False
    )
)

unit_price_outlier_summary

,OutlierRecords,MinimumPrice,MaximumPrice
ProductName,,,
Smart Watch,1588,95.52,114.99


### Initial Observations

- Quantity contains no statistical outliers under the IQR method.
- UnitPrice contains 1,588 potential outliers (3.18%).
- DiscountPct contains 12 potential outliers (0.02%).
- GrossSales contains 2,781 potential outliers (5.56%).
- DiscountAmount contains 3,108 potential outliers (6.22%).
- NetSales contains 2,803 potential outliers (5.61%).
- All 1,588 UnitPrice outliers belong to Smart Watch transactions.
- Their prices range from 95.52 to 114.99.
- These values may represent legitimate product-specific pricing rather than data-entry errors.
- Outlier treatment should consider product-level pricing patterns and business context.
- Statistical outliers are not automatically invalid records.
- Further business-level investigation is required before deciding whether any outliers should be treated or retained.

## 8. Consolidated Data Quality Summary

Summarise the identified data quality issues, their affected columns, and the observations that will guide the subsequent cleaning process.

In [36]:

quality_summary = pd.DataFrame({
    "Issue": [
        "Missing Values",
        "Exact Duplicates",
        "Inconsistent Categories",
        "Negative Unit Prices",
        "Zero Quantities",
        "Discounts Above 100%",
        "Date Inconsistencies",
        "Financial Calculation Errors",
        "Potential Statistical Outliers"
    ],
    "Affected Column(s)": [
        "Category, Region, PaymentMethod, ReturnDate",
        "All columns",
        "Category",
        "UnitPrice",
        "Quantity",
        "DiscountPct",
        "OrderDate, CustomerRegistrationDate, ReturnDate",
        "GrossSales, DiscountAmount, NetSales",
        "UnitPrice, GrossSales, DiscountAmount, NetSales"
    ],
    "Findings": [
        "Category: 90, Region: 80, PaymentMethod: 79 (require investigation); ReturnDate: 46,810 (expected for non-returned orders)",
        "100 duplicate occurrences",
        "28 variations representing 7 logical categories",
        "10 negative values",
        "15 zero values",
        "12 values above 100%",
        "No logical inconsistencies identified",
        "No calculation mismatches identified",
        "Potential outliers identified using the IQR method"
    ],
    "Requires Investigation": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "No",
        "No",
        "Yes"
    ]
})

quality_summary


,Issue,Affected Column(s),Findings,Requires Investigation
0,Missing Values,"Category, Region, PaymentMethod, ReturnDate","Category: 90, Region: 80, PaymentMethod: 79 (r...",Yes
1,Exact Duplicates,All columns,100 duplicate occurrences,Yes
2,Inconsistent Categories,Category,28 variations representing 7 logical categories,Yes
3,Negative Unit Prices,UnitPrice,10 negative values,Yes
4,Zero Quantities,Quantity,15 zero values,Yes
5,Discounts Above 100%,DiscountPct,12 values above 100%,Yes
6,Date Inconsistencies,"OrderDate, CustomerRegistrationDate, ReturnDate",No logical inconsistencies identified,No
7,Financial Calculation Errors,"GrossSales, DiscountAmount, NetSales",No calculation mismatches identified,No
8,Potential Statistical Outliers,"UnitPrice, GrossSales, DiscountAmount, NetSales",Potential outliers identified using the IQR me...,Yes


## Conclusion

The data quality assessment identified several issues requiring attention before analytical use, including missing values, duplicate records, inconsistent category formatting, invalid numerical values,
and potential statistical outliers.

Date consistency and financial calculation checks passed without identifying any logical inconsistencies or calculation mismatches.

The identified issues will be addressed in the subsequent data cleaning stage. The original raw dataset remains unchanged.